# Exploring the BPE Tokenizer

This notebook walks through how the BPE tokenizer trained on ML ArXiv papers works — what it does to words, how it handles unknown terms, and what the vocabulary looks like.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.data.tokenizer import load_tokenizer
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

tokenizer = load_tokenizer('../data/tokenizer.json')
vocab = tokenizer.get_vocab()  # dict: token_string -> token_id
print(f'Vocabulary size: {len(vocab):,}')

## 1. How individual words are tokenized

See how the tokenizer splits different words — common words become single tokens, rare or compound words get split.

In [ ]:
words = [
    # Common ML terms
    "attention", "transformer", "gradient", "backpropagation",
    "self-supervised", "cross-lingual", "hyperparameter",
    # Common English words
    "the", "and", "of", "learning", "network",
    # Rare / made-up words
    "supercalifragilistic", "neuromorphic", "LoRA", "GPT",
]

print(f"{'Word':<25} {'Tokens':<45} {'# tokens'}")
print('-' * 80)
for word in words:
    enc = tokenizer.encode(word)
    tokens = enc.tokens
    print(f"{word:<25} {str(tokens):<45} {len(tokens)}")

## 2. Tokenizing full sentences

In [ ]:
sentences = [
    "We propose a novel attention mechanism for language modeling.",
    "The transformer architecture has revolutionized natural language processing.",
    "Our method achieves state-of-the-art results on standard benchmarks.",
]

for sentence in sentences:
    enc = tokenizer.encode(sentence)
    print(f"Text:   {sentence}")
    print(f"Tokens: {enc.tokens}")
    print(f"IDs:    {enc.ids}")
    print(f"Count:  {len(enc.ids)} tokens for {len(sentence.split())} words")
    print()

## 3. Token length distribution

How many characters are in the average token? Short tokens = fine-grained splits. Long tokens = common phrases encoded efficiently.

In [ ]:
# Get all tokens and their lengths
all_tokens = list(vocab.keys())

# Filter out special tokens and byte-level artifacts for cleaner analysis
regular_tokens = [t for t in all_tokens if not t.startswith('[') and len(t) > 0]

# Decode byte-level encoding (Ġ = space prefix in GPT-style BPE)
decoded_lengths = [len(t.replace('Ġ', ' ').replace('Ċ', '\n')) for t in regular_tokens]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of token lengths
axes[0].hist(decoded_lengths, bins=range(1, 25), color='#2563eb', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('Token length (characters)', fontsize=12)
axes[0].set_ylabel('Number of tokens', fontsize=12)
axes[0].set_title('Token Length Distribution', fontsize=13, fontweight='bold')
axes[0].yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
axes[0].grid(True, alpha=0.3)

# Cumulative distribution
sorted_lengths = sorted(decoded_lengths)
cumulative = np.arange(1, len(sorted_lengths) + 1) / len(sorted_lengths)
axes[1].plot(sorted_lengths, cumulative * 100, color='#2563eb', linewidth=2)
axes[1].set_xlabel('Token length (characters)', fontsize=12)
axes[1].set_ylabel('Cumulative % of vocabulary', fontsize=12)
axes[1].set_title('Cumulative Token Length', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].axhline(50, color='gray', linestyle='--', alpha=0.5, label='50th percentile')
axes[1].axhline(90, color='gray', linestyle=':', alpha=0.5, label='90th percentile')
axes[1].legend()

plt.tight_layout()
plt.savefig('../plots/token_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

import os; os.makedirs('../plots', exist_ok=True)

print(f"Average token length: {np.mean(decoded_lengths):.2f} chars")
print(f"Median token length:  {np.median(decoded_lengths):.0f} chars")
print(f"Longest token:        {max(decoded_lengths)} chars")

## 4. Most common tokens in the vocabulary

Tokens with low IDs were learned first (most frequent merges). High IDs = rare tokens.

In [ ]:
# Sort by token ID (low ID = learned early = more common)
sorted_vocab = sorted(vocab.items(), key=lambda x: x[1])

print("First 50 tokens (most common / learned first):")
print([t for t, _ in sorted_vocab[:50]])

print("\nLast 20 tokens (rarest / learned last):")
print([t for t, _ in sorted_vocab[-20:]])

## 5. Tokens per word ratio on real abstracts

How many tokens does a typical abstract use? This tells us how efficiently the tokenizer compresses the domain.

In [ ]:
from datasets import load_dataset

print("Loading a sample of abstracts...")
ds = load_dataset("CShorten/ML-ArXiv-Papers", split="train[:500]")

token_counts = []
word_counts = []
ratios = []

for ex in ds:
    text = (ex['title'] + '\n' + ex['abstract']).strip()
    n_tokens = len(tokenizer.encode(text).ids)
    n_words = len(text.split())
    token_counts.append(n_tokens)
    word_counts.append(n_words)
    if n_words > 0:
        ratios.append(n_tokens / n_words)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(token_counts, bins=40, color='#2563eb', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('Tokens per abstract', fontsize=12)
axes[0].set_ylabel('Number of abstracts', fontsize=12)
axes[0].set_title('Tokens per Abstract (sample of 500)', fontsize=13, fontweight='bold')
axes[0].axvline(512, color='#dc2626', linestyle='--', label='seq_len=512')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(ratios, bins=30, color='#16a34a', edgecolor='white', alpha=0.85)
axes[1].set_xlabel('Tokens per word', fontsize=12)
axes[1].set_ylabel('Number of abstracts', fontsize=12)
axes[1].set_title('Tokenization Efficiency (tokens/word)', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../plots/tokenizer_stats.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Average tokens per abstract: {np.mean(token_counts):.0f}")
print(f"Average tokens per word:     {np.mean(ratios):.2f}")
print(f"% abstracts fitting in 512 tokens: {sum(t <= 512 for t in token_counts) / len(token_counts) * 100:.1f}%")

## 6. Round-trip test

Encode text → decode back → verify it matches the original. A correct tokenizer should always be lossless.

In [ ]:
test_texts = [
    "We propose a novel self-supervised learning framework.",
    "GPT-4 achieves 90% accuracy on MMLU benchmarks.",
    "The model uses 175 billion parameters trained on 300B tokens.",
]

print("Round-trip encode → decode test:")
for text in test_texts:
    ids = tokenizer.encode(text).ids
    decoded = tokenizer.decode(ids)
    match = text.strip() == decoded.strip()
    status = '✓' if match else '✗'
    print(f"  {status}  Original: {text!r}")
    if not match:
        print(f"     Decoded: {decoded!r}")